In [3]:
# %pip install -q datasets pandas scikit-learn

In [1]:
from datasets import load_dataset

ds = load_dataset("PolyAI/banking77", revision="refs/pr/6", cache_dir="./hf_cache")
print(ds)
print(ds["train"][0])

c:\Users\Hoang\miniconda3\envs\nlp_lab2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
{'text': 'I am still waiting on my card?', 'label': 11}


In [2]:
import pandas as pd

train_df = pd.DataFrame(ds["train"])
test_df = pd.DataFrame(ds["test"])

print(train_df.head())
print(train_df["label"].nunique())
print(train_df["label"].value_counts().sort_index().head())

                                                text  label
0                     I am still waiting on my card?     11
1  What can I do if my card still hasn't arrived ...     11
2  I have been waiting over a week. Is the card s...     11
3  Can I track my card while it is in the process...     11
4  How do I know if I will get my card, or if it ...     11
77
label
0    159
1    110
2    126
3     87
4    127
Name: count, dtype: int64


In [3]:
label_names = ds["train"].features["label"].names
print("Số intent:", len(label_names))
print(label_names[:20])

Số intent: 77
['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge']


In [4]:
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

train_df["label_name"] = train_df["label"].map(id2label)
test_df["label_name"] = test_df["label"].map(id2label)

print(train_df[["text", "label", "label_name"]].head())

                                                text  label    label_name
0                     I am still waiting on my card?     11  card_arrival
1  What can I do if my card still hasn't arrived ...     11  card_arrival
2  I have been waiting over a week. Is the card s...     11  card_arrival
3  Can I track my card while it is in the process...     11  card_arrival
4  How do I know if I will get my card, or if it ...     11  card_arrival


In [5]:
selected_labels = [
    "card_arrival",
    "card_not_working",
    "declined_card_payment",
    "card_payment_not_recognised",
    "card_payment_fee_charged",
    "card_swallowed",
    "lost_or_stolen_card",
    "pending_card_payment",
    "card_delivery_estimate",
    "activate_my_card"
]

selected_ids = [label2id[name] for name in selected_labels]

train_sub = train_df[train_df["label"].isin(selected_ids)].copy()
test_sub = test_df[test_df["label"].isin(selected_ids)].copy()

print(train_sub["label_name"].value_counts())
print(test_sub["label_name"].value_counts())

label_name
card_payment_fee_charged       187
card_payment_not_recognised    168
activate_my_card               159
pending_card_payment           159
declined_card_payment          153
card_arrival                   153
card_delivery_estimate         112
card_not_working               112
lost_or_stolen_card             82
card_swallowed                  61
Name: count, dtype: int64
label_name
card_arrival                   40
card_delivery_estimate         40
card_not_working               40
lost_or_stolen_card            40
card_payment_fee_charged       40
card_payment_not_recognised    40
pending_card_payment           40
declined_card_payment          40
card_swallowed                 40
activate_my_card               40
Name: count, dtype: int64


In [6]:
new_label_names = sorted(train_sub["label_name"].unique())
new_label2id = {name: i for i, name in enumerate(new_label_names)}
new_id2label = {i: name for name, i in new_label2id.items()}

train_sub["new_label"] = train_sub["label_name"].map(new_label2id)
test_sub["new_label"] = test_sub["label_name"].map(new_label2id)

print(train_sub[["text", "label_name", "new_label"]].head())

                                                text    label_name  new_label
0                     I am still waiting on my card?  card_arrival          1
1  What can I do if my card still hasn't arrived ...  card_arrival          1
2  I have been waiting over a week. Is the card s...  card_arrival          1
3  Can I track my card while it is in the process...  card_arrival          1
4  How do I know if I will get my card, or if it ...  card_arrival          1


In [7]:
import os

os.makedirs("sample_data", exist_ok=True)

In [8]:
train_out = train_sub[["text", "new_label", "label_name"]].rename(
    columns={"new_label": "label"}
)
test_out = test_sub[["text", "new_label", "label_name"]].rename(
    columns={"new_label": "label"}
)

train_out.to_csv("sample_data/train.csv", index=False)
test_out.to_csv("sample_data/test.csv", index=False)

print("Saved train/test CSV to sample_data/")
print(train_out.head())

Saved train/test CSV to sample_data/
                                                text  label    label_name
0                     I am still waiting on my card?      1  card_arrival
1  What can I do if my card still hasn't arrived ...      1  card_arrival
2  I have been waiting over a week. Is the card s...      1  card_arrival
3  Can I track my card while it is in the process...      1  card_arrival
4  How do I know if I will get my card, or if it ...      1  card_arrival


In [9]:
import json

label_config = {
    "label2id": new_label2id,
    "id2label": {str(k): v for k, v in new_id2label.items()},
    "selected_labels": selected_labels,
}

os.makedirs("configs", exist_ok=True)

with open("configs/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(label_config, f, ensure_ascii=False, indent=2)

In [10]:
print("Train shape:", train_out.shape)
print("Test shape:", test_out.shape)

print("\nTrain distribution:")
print(train_out["label_name"].value_counts())

print("\nTest distribution:")
print(test_out["label_name"].value_counts())

print("\nMissing values:")
print(train_out.isnull().sum())
print(test_out.isnull().sum())

Train shape: (1346, 3)
Test shape: (400, 3)

Train distribution:
label_name
card_payment_fee_charged       187
card_payment_not_recognised    168
activate_my_card               159
pending_card_payment           159
declined_card_payment          153
card_arrival                   153
card_delivery_estimate         112
card_not_working               112
lost_or_stolen_card             82
card_swallowed                  61
Name: count, dtype: int64

Test distribution:
label_name
card_arrival                   40
card_delivery_estimate         40
card_not_working               40
lost_or_stolen_card            40
card_payment_fee_charged       40
card_payment_not_recognised    40
pending_card_payment           40
declined_card_payment          40
card_swallowed                 40
activate_my_card               40
Name: count, dtype: int64

Missing values:
text          0
label         0
label_name    0
dtype: int64
text          0
label         0
label_name    0
dtype: int64
